In [1]:
## WORKSPACE :
# -----------
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md


from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

INPUT = Path("/kaggle/input/")
DATA_ROOT = INPUT / "competitions/rsna-knee-abnormality-detection"
WORKING = Path("/kaggle/working/")
np.random.seed(202609)

"""
DATA HIERARCHY: 
***************

[*] TRAINING SET:
    -------------
    - train.csv : 
        The outermost layer, where each row is a complete study with a unique StudyInstanceUID and a multilingual narrative Report. 
        There are 12 label columns which will be used as target columns of the image classifier, 
        but only a fraction of studies are pre-labelled.
        
        
    - train_series.csv : 
        Each study consists of multiple MRI series;
        e.g. side-to-side (sagittal), top-to-bottom (axial), front-to-back (coronal) with different fluid and fat sensitivity settings.
        Here, each row is a unique (StudyInstanceUID, SeriesInstanceUID) with its corresponding flags for MRI settings.
        
    - train_series : WARNING : 500GB+
        This is where the actual MRI images are stored.
        - One folder for each StudyInstanceUID in train.csv.
            - One subfolder for each (StudyInstanceUID, SeriesInstanceUID) in train_series.csv.
              Each such subfolder consists of multiple DICOM files;
                - one file represents an MRI slice.
        
[*] TEST SET: For the test set, we do not have any written narrative. The image-data hierarchy remains the same.
    ---------

OBJECTIVE: Predict the probability of 12 abnormality classes given MRI imagery.
**********
[Step 1] Populate the TARGETS based on the radiology Report.
[Step 2] Train a multi-label classifier to predict the TARGETS from the corresponding DICOM images.

"""

TARGETS = pd.read_csv(
    DATA_ROOT / "sample_submission.csv",
    nrows=0, index_col="StudyInstanceUID",
).columns.tolist()

train = pd.read_csv(
    DATA_ROOT / "train.csv",
    index_col="StudyInstanceUID",
    dtype={"Report":'string'}|{col:np.float32 for col in TARGETS},
    # we will use soft labels instead of hard 0-1 labeling
)
train = train.set_index(
    pd.MultiIndex.from_arrays(
        [train[TARGETS].notna().any(axis=1).values, train.index],
        names=["label_mask", "StudyInstanceUID"],
    )
).sort_index()
LABELLED = train.loc[(True,), :].astype({col:bool for col in TARGETS})
#LABELLED.to_csv(WORKING / "gold_standards.csv")
n_LABELLED = len(LABELLED)

train_series = pd.read_csv(
    DATA_ROOT / "train_series.csv",
    index_col=["StudyInstanceUID", "SeriesInstanceUID"],
    dtype='category', 
    # only 2-3 unique labels for each column of len ~25k
).sort_index()


print("\ntrain: ", train.shape)
display(train)
print("\ntrain_series: ", train_series.shape)
display(train_series)
print("\ntargets: ", TARGETS)
print("\nStudies with labels: ", n_LABELLED)
display(
    (
        100 * pd.DataFrame(
            [ 
                LABELLED[TARGETS].sum(axis=0),
                (~LABELLED[TARGETS]).sum(axis=0),
            ],
            index= ["label_positive (%)", "label_negative (%)"],
            # positive and negative labels not necessarily adds up to 100%, 
            # if NaN is there
        ) / n_LABELLED
    ).round(2)
)
print("\nStudies without labels: ", len(train) - n_LABELLED)
print("\nReport word-counts: ", train["Report"].str.len().describe().astype(np.int32).to_dict())



train:  (4407, 13)


Report  \
label_mask StudyInstanceUID                                                                                        
False      1.2.826.0.1.3680043.8.498.100048732290990538690...  Técnica: RMN de la rodilla. Resultados: Rotura...   
           1.2.826.0.1.3680043.8.498.100049459274726560271...  [DATE]: * MR Knie Rechts 15ch AA Klinische Inl...   
           1.2.826.0.1.3680043.8.498.100092786926066315735...  Hallazgos:\nNo hay alteraciones en significati...   
           1.2.826.0.1.3680043.8.498.100096392031707502741...   In the medial compartment, the meniscus is no...   
           1.2.826.0.1.3680043.8.498.100136637424007360294...  CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...   
...                                                                                                          ...   
True       1.2.826.0.1.3680043.8.498.869686002397243106789...  Right knee pain. R/O ACL insufficiency. \nMRI ...   
           1.2.826.0.1.3680043.8.498.880774186393011744099...  Uznapredovale goanrtrotske promjene uz opsežan...   
           1.2.826.0.1.3680043.8.498.902835653810420817685...  Técnica: RMN de la rodilla. Resultados: Cambio...   
           1.2.826.0.1.3680043.8.498.944337534718903061080...  No significant knee effusion. Normal medial an...   
           1.2.826.0.1.3680043.8.498.972747202576345840715...  Stigmata der Inaktivitätsosteopenie sowie verb...   

                                                               ACL  MCL  \
label_mask StudyInstanceUID                                               
False      1.2.826.0.1.3680043.8.498.100048732290990538690...  NaN  NaN   
           1.2.826.0.1.3680043.8.498.100049459274726560271...  NaN  NaN   
           1.2.826.0.1.3680043.8.498.100092786926066315735...  NaN  NaN   
           1.2.826.0.1.3680043.8.498.100096392031707502741...  NaN  NaN   
           1.2.826.0.1.3680043.8.498.100136637424007360294...  NaN  NaN   
...                                                            ...  ...   
True       1.2.826.0.1.3680043.8.498.869686002397243106789...  1.0  0.0   
           1.2.826.0.1.3680043.8.498.880774186393011744099...  1.0  0.0   
           1.2.826.0.1.3680043.8.498.902835653810420817685...  0.0  0.0   
           1.2.826.0.1.3680043.8.498.944337534718903061080...  0.0  0.0   
           1.2.826.0.1.3680043.8.498.972747202576345840715...  1.0  0.0   

                                                               Medial Meniscus  \
label_mask StudyInstanceUID                                                      
False      1.2.826.0.1.3680043.8.498.100048732290990538690...              NaN   
           1.2.826.0.1.3680043.8.498.100049459274726560271...              NaN   
           1.2.826.0.1.3680043.8.498.100092786926066315735...              NaN   
           1.2.826.0.1.3680043.8.498.100096392031707502741...              NaN   
           1.2.826.0.1.3680043.8.498.100136637424007360294...              NaN   
...                                                                        ...   
True       1.2.826.0.1.3680043.8.498.869686002397243106789...              1.0   
           1.2.826.0.1.3680043.8.498.880774186393011744099...              1.0   
           1.2.826.0.1.3680043.8.498.902835653810420817685...              0.0   
           1.2.826.0.1.3680043.8.498.944337534718903061080...              0.0   
           1.2.826.0.1.3680043.8.498.972747202576345840715...              1.0   

                                                               Lateral Meniscus  \
label_mask StudyInstanceUID                                                       
False      1.2.826.0.1.3680043.8.498.100048732290990538690...               NaN   
           1.2.826.0.1.3680043.8.498.100049459274726560271...               NaN   
           1.2.826.0.1.3680043.8.498.100092786926066315735...               NaN   
           1.2.826.0.1.3680043.8.498.100096392031707502741...               NaN   
           1.2.826.0.1.3680043.8.498.1001366374240073602


train_series:  (24371, 3)


Fluid_Sensitive  \
StudyInstanceUID                                   SeriesInstanceUID                                                    
1.2.826.0.1.3680043.8.498.100048732290990538690... 1.2.826.0.1.3680043.8.498.123431101950362134834...               1   
                                                   1.2.826.0.1.3680043.8.498.138212297449972206415...               1   
                                                   1.2.826.0.1.3680043.8.498.230848365367225952758...               0   
                                                   1.2.826.0.1.3680043.8.498.407342061024587230961...               1   
                                                   1.2.826.0.1.3680043.8.498.757148999972036157840...               0   
...                                                                                                               ...   
1.2.826.0.1.3680043.8.498.999576112073130986961... 1.2.826.0.1.3680043.8.498.543090017839016684838...               1   
                                                   1.2.826.0.1.3680043.8.498.628702981296394013441...               1   
                                                   1.2.826.0.1.3680043.8.498.750798929395794071069...               0   
                                                   1.2.826.0.1.3680043.8.498.753175095197121489120...               1   
                                                   1.2.826.0.1.3680043.8.498.957222785063165420178...               1   

                                                                                                      Fat_Suppression  \
StudyInstanceUID                                   SeriesInstanceUID                                                    
1.2.826.0.1.3680043.8.498.100048732290990538690... 1.2.826.0.1.3680043.8.498.123431101950362134834...               1   
                                                   1.2.826.0.1.3680043.8.498.138212297449972206415...               1   
                                                   1.2.826.0.1.3680043.8.498.230848365367225952758...               0   
                                                   1.2.826.0.1.3680043.8.498.407342061024587230961...               1   
                                                   1.2.826.0.1.3680043.8.498.757148999972036157840...               0   
...                                                                                                               ...   
1.2.826.0.1.3680043.8.498.999576112073130986961... 1.2.826.0.1.3680043.8.498.543090017839016684838...               1   
                                                   1.2.826.0.1.3680043.8.498.628702981296394013441...               1   
                                                   1.2.826.0.1.3680043.8.498.750798929395794071069...               0   
                                                   1.2.826.0.1.3680043.8.498.753175095197121489120...               1   
                                                   1.2.826.0.1.3680043.8.498.957222785063165420178...               1   

                                                                                                      Anatomical_Plane  
StudyInstanceUID                                   SeriesInstanceUID                                                    
1.2.826.0.1.3680043.8.498.100048732290990538690... 1.2.826.0.1.3680043.8.498.123431101950362134834...         Sagittal  
                                                   1.2.826.0.1.3680043.8.498.138212297449972206415...            Axial  
                                                   1.2.826.0.1.3680043.8.498.230848365367225952758...          Coronal  
                                                   1.2.826.0.1.3680043.8.498.407342061024587230961...          Coronal  
                                                   1.2.826.0.1.3680043.8.498.757148999972036157840...         Sagittal  
...                                                                                                        


targets:  ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

Studies with labels:  58


,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
label_positive (%),41.38,15.52,44.83,39.66,25.86,18.97,36.21,60.34,46.55,20.69,32.76,31.03
label_negative (%),58.62,84.48,55.17,60.34,74.14,81.03,63.79,39.66,53.45,79.31,67.24,68.97



Studies without labels:  4349

Report word-counts:  {'count': 4407, 'mean': 1097, 'std': 693, 'min': 52, '25%': 587, '50%': 977, '75%': 1459, 'max': 4743}


In [3]:
%%writefile /kaggle/working/report_labeler.py

"""
Runs INSIDE a Kaggle kernel. 
"""

import os
from pathlib import Path
import pandas as pd

# Quiet down vLLM/transformers/huggingface_hub's own console output
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")   
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")  
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")    




# =========================================================
# Config
# =========================================================

MODEL_NAME = "Qwen/Qwen3-4B"
# MODEL CHOICE: ref- https://arxiv.org/html/2505.09388v1
# *************
#   - Apache-2.0 license, no non-commercial restriction
#   - officially covers 100+ languages [includes all major target languages in the underlying project]
#   - fits comfortably on a T4 GPU [CRUCIAL for Kaggle environment]

TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
OUTPUT_JSONL = "/kaggle/working/report_labels_raw.jsonl"
TIME_BUDGET_SECONDS = 8 * 3600 # Kaggle's own PLATFORM session cap (applies to any running kernel, historically ~9-12h for GPU) 
MAX_NEW_TOKENS, RETRY_MAX_NEW_TOKENS = 1200, 2000 # only used for requests that hit MAX_NEW_TOKENS without finishing
MAX_MODEL_LEN, MAX_SEQ_LEN = 8192, 8192

ENABLE_THINKING = False
# deterministic extraction — no need for Qwen3's chain-of-thought mode.
# avoid latency across ~4,400 reports for no benefit on a fixed-schema extraction task

BATCH_SIZE = 200  
# StudyInstanceUID per vLLM generate() call — controls checkpoint-write frequency and
# how much work is lost if a session dies mid-batch

FINDINGS = [
    "ACL", "MCL", 
    "Medial Meniscus", "Lateral Meniscus", 
    "Medial OA", "Lateral OA", "PF OA", 
    "Effusion", "Synovitis", 
    "Baker's", 
    "Contusion", 
    "Fracture",
] # the target classes

SYSTEM_PROMPT = """You are a musculoskeletal radiology assistant. You will be given the full text of one knee MRI
report, which may be written in English, Spanish, Turkish, Croatian, Dutch, German, Greek, or
Bulgarian (or occasionally another language). The report describes one knee MRI study.

Your task: for each of the 12 findings listed below, determine whether the report explicitly
supports it being present, explicitly supports it being absent, or does not address it at all.
Extract findings based ONLY on what the report states. Do not infer beyond the text. Do not use
outside medical knowledge to guess at what "should" be present.

Read the ENTIRE report before answering, including any Findings, Impression, or Conclusion section 
— the Impression may confirm, add to, or occasionally omit findings mentioned in the body.
When body and impression differ, use the more complete/explicit statement to determine the state and quote the supporting statement as evidence.

## The 12 findings

1. ACL (anterior cruciate ligament) — tear/injury/sprain
2. MCL (medial collateral ligament) — tear/injury/sprain
3. Medial Meniscus — tear only (degeneration/signal change alone does NOT count)
4. Lateral Meniscus — tear only (degeneration/signal change alone does NOT count)
5. Medial OA — cartilage loss/chondrosis/chondromalacia/osteophytes in MEDIAL femorotibial compartment (location, not the word "arthritis", decides this)
6. Lateral OA — cartilage loss/chondrosis/chondromalacia/osteophytes in LATERAL femorotibial compartment (location, not the word "arthritis", decides this)
7. PF OA — cartilage loss/chondrosis/chondromalacia/osteophytes in PATELLOFEMORAL compartment/trochlea/patellar facets (location, not the word "arthritis", decides this)
8. Effusion — joint effusion/fluid
9. Synovitis — synovial thickening/hypertrophy/proliferation (Hoffa fat-pad/"hoffitis" language is NOT evidence either way)
10. Baker's — popliteal cyst / Baker's cyst
11. Contusion — bone bruise/marrow edema ONLY if framed as traumatic/impact; degenerative or fracture-adjacent edema does NOT count
12. Fracture — any fracture line/impaction/avulsion/subchondral insufficiency fracture

## Ligament group-clearance

- CRUCIATE-only statement ("cruciate ligaments preserved", "ACL and PCL intact") → clears ACL only
- COLLATERAL-only statement ("collateral ligaments normal") → clears MCL only
- Combined cruciate+collateral statement → clears both ACL and MCL
- A specific ligament named abnormal elsewhere overrides its clearance

## States (pick exactly one per finding)

- explicit_positive: clearly stated as present. 
  Severity does NOT change this state — "trace", "mild", "grade 1", "minimal" are still explicit_positive.
  Do not judge whether a severity would count toward the final label — that's a downstream decision, not yours.
- explicit_negative: clearly stated as absent/normal/intact (directly or via group-clearance)
- hedged_positive: uncertain language suggesting presence ("possible", "suspicious for", "cannot exclude", "suggestive of")
- hedged_negative: uncertain language suggesting absence, no firm statement either way
- unmentioned: not addressed at all


## Laterality

Report should describe ONE knee. 
If it clearly describes two knees separately in two different sections, 
set "ambiguous_laterality": true and extract findings only from the FIRST-listed section — ignore
the second knee's findings entirely; the ambiguous_laterality flag alone is the signal for this case. 
Evidence stays a plain exact quote from that FIRST section.

## Evidence

For every non-"unmentioned" finding, give the shortest EXACT quote (a verbatim substring of the report, under 15 words) 
justifying the state. Do not paraphrase, reword, or translate — copy the text exactly as it appears in the report, 
in the report's original language.

## Output — ONLY this JSON, no fences, no commentary:

{
  "ambiguous_laterality": false,
  "findings": {
    "ACL": {"state": "explicit_positive", "evidence": "suspect grade 1 injury of ACL"},
    "MCL": {"state": "explicit_negative", "evidence": "collateral ligaments intact"},
    "Medial Meniscus": {"state": "unmentioned", "evidence": null},
    "Lateral Meniscus": {"state": "explicit_positive", "evidence": "tear of lateral meniscus"},
    "Medial OA": {"state": "unmentioned", "evidence": null},
    "Lateral OA": {"state": "unmentioned", "evidence": null},
    "PF OA": {"state": "unmentioned", "evidence": null},
    "Effusion": {"state": "explicit_positive", "evidence": "small joint effusion"},
    "Synovitis": {"state": "unmentioned", "evidence": null},
    "Baker's": {"state": "explicit_negative", "evidence": "no Baker's cyst"},
    "Contusion": {"state": "unmentioned", "evidence": null},
    "Fracture": {"state": "unmentioned", "evidence": null}
  }
}


All 12 keys required every time, even if unmentioned."""


# =========================================================
# Few-shot examples

# Recent studies have demonstrated that few-shot approaches can significantly outperform
# zero-shot approaches in similar domain tasks.
# ref: 
# ----
# - https://arxiv.org/pdf/2511.10658
# - https://pubs.rsna.org/doi/pdf/10.1148/ryai.240551

# =========================================================

FEW_SHOT_EXAMPLES = [
    (# EN: ligament clearance + meniscus tear-word rule [[UID 1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238]]
        """ In the medial compartment, there is longitudinal vertical tear at the body and posterior horn of the medial meniscus. There is no focal chondrosis or chondral injury. 
 The lateral compartment, the meniscus is not torn and there is no focal chondrosis or chondral injury. 
 The patellofemoral joint, the patellar and trochlear cartilage appears congruent without chondral lesion. The medial patellofemoral ligament is intact.  
 In the intercondylar notch, there is complete tear of the anterior cruciate ligament, at mid substance. The posterior cruciate ligament is intact. There is anterior tibial translation.   
 The medial and lateral collateral ligament complexes are intact. The extensor mechanism is normal.  
 No knee effusion is present. The surrounding musculature is normal. There is no fracture or bone contusion.   -- Longitudinal vertical tear at the body and posterior horn of the medial meniscus. 
 -- Complete tear of the anterior cruciate ligament, at mid substance. """,
        """{
            "ambiguous_laterality": false,
            "findings": {
                "ACL": {"state": "explicit_positive", "evidence": "complete tear of the anterior cruciate ligament, at mid substance"},
                "MCL": {"state": "explicit_negative", "evidence": "medial and lateral collateral ligament complexes are intact"},
                "Medial Meniscus": {"state": "explicit_positive", "evidence": "longitudinal vertical tear at the body and posterior horn"},
                "Lateral Meniscus": {"state": "explicit_negative", "evidence": "the meniscus is not torn"},
                "Medial OA": {"state": "unmentioned", "evidence": null},
                "Lateral OA": {"state": "unmentioned", "evidence": null},
                "PF OA": {"state": "explicit_negative", "evidence": "patellar and trochlear cartilage appears congruent without chondral lesion"},
                "Effusion": {"state": "explicit_negative", "evidence": "no knee effusion is present"},
                "Synovitis": {"state": "unmentioned", "evidence": null},
                "Baker's": {"state": "unmentioned", "evidence": null},
                "Contusion": {"state": "explicit_negative", "evidence": "no fracture or bone contusion"},
                "Fracture": {"state": "explicit_negative", "evidence": "no fracture or bone contusion"}
            }
        }""",
    ),
    (# TR: hedged_positive, text-faithful even where report/gold diverge [[UID 1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711]]
        """Diz eklemi içi sıvı miktarı hafif derecede artmış. Lateral femoral kondilin ağırlık taşıyan yüzeyinin anteriyor yarısındaki subkondral kırığı kemik iliği ödemi çevreliyor. Tibyal plato posterolateral köşesinde pivot kayması ile ilintili kemik kontüzyonu mevcut. Ön çapraz bağdaki laksite artışı ve ödem yüksek olasılıkla tam kat-tam kata yakın yırtık ile birlikte. Posteriyor oblik ve oblik popliteal bağlar ödemli ancak menisküs yırtığı eşlik etmiyor. Posteriyor medyal köşe yapıları bunların dışında korunmuş. Arka çapraz ve yan bağlar normal. Posterolateral köşe yapıları korunmuş. Medyal menisküs ve lateral menisküs normal. Eklem kıkırdakları ve kemikler bunların dışında normal. Ekstensör mekanizma korunmuş (kuadriseps tendonu ve patellar tendon, patella ve yerleşimi, femoral troklear oluk ve patellofemoral bağlar normal).""",
        """{
            "ambiguous_laterality": false,
            "findings": {
                "ACL": {"state": "hedged_positive", "evidence": "yüksek olasılıkla tam kat-tam kata yakın yırtık ile birlikte"},
                "MCL": {"state": "explicit_negative", "evidence": "yan bağlar normal"},
                "Medial Meniscus": {"state": "explicit_negative", "evidence": "Medyal menisküs ve lateral menisküs normal"},
                "Lateral Meniscus": {"state": "explicit_negative", "evidence": "Medyal menisküs ve lateral menisküs normal"},
                "Medial OA": {"state": "unmentioned", "evidence": null},
                "Lateral OA": {"state": "unmentioned", "evidence": null},
                "PF OA": {"state": "unmentioned", "evidence": null},
                "Effusion": {"state": "explicit_positive", "evidence": "sıvı miktarı hafif derecede artmış"},
                "Synovitis": {"state": "unmentioned", "evidence": null},
                "Baker's": {"state": "unmentioned", "evidence": null},
                "Contusion": {"state": "explicit_positive", "evidence": "pivot kayması ile ilintili kemik kontüzyonu mevcut"},
                "Fracture": {"state": "explicit_positive", "evidence": "subkondral kırığı"}
            }
        }""",
    ),
    (# EN: correct "unmentioned" on a finding gold later scores positive [[UID 1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312]]
        """No significant knee effusion. Normal medial and lateral menisci. No evidence of tears.  Normal ACL, PCL, MCL and LCL. Normal extensor mechanism.  The tibiofemoral and patellofemoral articular cartilage is normal. Normal visualized bones and surrounding soft tissues.  Conclusion: No evidence of meniscal tears or ligamentous injury.""",
        """{
            "ambiguous_laterality": false,
            "findings": {
                "ACL": {"state": "explicit_negative", "evidence": "Normal ACL, PCL, MCL and LCL"},
                "MCL": {"state": "explicit_negative", "evidence": "Normal ACL, PCL, MCL and LCL"},
                "Medial Meniscus": {"state": "explicit_negative", "evidence": "normal medial and lateral menisci"},
                "Lateral Meniscus": {"state": "explicit_negative", "evidence": "normal medial and lateral menisci"},
                "Medial OA": {"state": "explicit_negative", "evidence": "tibiofemoral and patellofemoral articular cartilage is normal"},
                "Lateral OA": {"state": "explicit_negative", "evidence": "tibiofemoral and patellofemoral articular cartilage is normal"},
                "PF OA": {"state": "explicit_negative", "evidence": "tibiofemoral and patellofemoral articular cartilage is normal"},
                "Effusion": {"state": "explicit_negative", "evidence": "No significant knee effusion"},
                "Synovitis": {"state": "unmentioned", "evidence": null},
                "Baker's": {"state": "unmentioned", "evidence": null},
                "Contusion": {"state": "unmentioned", "evidence": null},
                "Fracture": {"state": "unmentioned", "evidence": null}
            }
        }""",
    ),
    (# BG: meniscus group-clearance + broad all-compartment cartilage clearance [[UID 1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803]]
        """МР находка: МР данни за фрактура на тибията в областта на интеркондиларните еминенции. Фрактурата е без данни за значима дислокация. Визуализира се контузионен костномозъчен едем в тибията срединно и към медиалното тибиално плато. Минимален субхондрален едем се изобразява и по медиалната повърхност на пателата. Хрущялът, покриващ ставните повърхности на костите е съхранен. Кръстните връзки са със запазена цялост. Нормално представени менискуси. Хрущял на пателата, ретинакулуми – нормално представени. Insall-Salvati ratio – 1,3. Нормално изобразяване на сухожилието на пателата и на м. квадрицепс феморис. Наличие на минимален ставен излив. Заключение:  МР данни за фрактура на тибията в областта на интеркондиларните еминенции, без значима дислокация. Контузионен костномозъчен едем. Минимален ставен излив. """,
        """{
            "ambiguous_laterality": false,
            "findings": {
                "ACL": {"state": "explicit_negative", "evidence": "Кръстните връзки са със запазена цялост"},
                "MCL": {"state": "unmentioned", "evidence": null},
                "Medial Meniscus": {"state": "explicit_negative", "evidence": "Нормално представени менискуси"},
                "Lateral Meniscus": {"state": "explicit_negative", "evidence": "Нормално представени менискуси"},
                "Medial OA": {"state": "explicit_negative", "evidence": "Хрущялът, покриващ ставните повърхности на костите е съхранен"},
                "Lateral OA": {"state": "explicit_negative", "evidence": "Хрущялът, покриващ ставните повърхности на костите е съхранен"},
                "PF OA": {"state": "explicit_negative", "evidence": "Хрущял на пателата, ретинакулуми – нормално представени"},
                "Effusion": {"state": "explicit_positive", "evidence": "Наличие на минимален ставен излив"},
                "Synovitis": {"state": "unmentioned", "evidence": null},
                "Baker's": {"state": "unmentioned", "evidence": null},
                "Contusion": {"state": "explicit_positive", "evidence": "контузионен костномозъчен едем"},
                "Fracture": {"state": "explicit_positive", "evidence": "фрактура на тибията в областта на интеркондиларните еминенции, без значима дислокация"}
            }
        }""",
    ),
    (# EN: compartment-based OA mapping, mild stays explicit_positive [[UID 1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052]]
        """Exam Type: MRI KNEE RIGHT WO CONTRAST
Exam Date and Time: [DATE] [TIME]
Indication: 2 years of pain with walking up and down the stairs twisting and squatting.
Comparison: No relevant prior studies available for comparison.
 
TECHNIQUE:
MRI of the right knee was performed on a 1.5 Tesla system body coil in three planes (axial, sagittal, and coronal), using a standard non-contrast protocol.
 
FINDINGS:
OSSEOUS STRUCTURES:
Alignment is anatomic. No acute fracture or bone bruise. Background marrow signal is normal without infiltrative marrow process. No aggressive osseous lesions. Tricompartmental marginal osteophytes
 
JOINT SPACE:
Small joint effusion. No intra-articular bodies. No Baker's cyst.
 
MEDIAL COMPARTMENT:
Medial meniscus: Extensive complete tearing of the body, posterior horn, and posterior root medial meniscus with severe extrusion.
Medial compartment cartilage: Full thickness cartilage loss along the medial femoral condyle and medial tibial plateau with subchondral cystic change and marrow edema.
 
LATERAL COMPARTMENT:
Lateral meniscus: Degenerative signal throughout the lateral meniscus without surfacing tear.
Lateral compartment cartilage: Mild cartilage heterogeneity and fissuring.
 
PATELLOFEMORAL COMPARTMENT:
Patellofemoral compartment cartilage: High-grade cartilage loss along the medial patellar facet. High grade cartilage loss at lateral trochlea. 
 
CRUCIATE LIGAMENTS:
Anterior cruciate ligament (ACL): Normal with both bundles intact.
Posterior cruciate ligament (PCL): Normal.
 
COLLATERAL LIGAMENTS & POSTEROLATERAL CORNER:
Medial collateral ligament (MCL): Normal.
Lateral collateral ligament (LCL) complex: Normal.
Posterolateral corner structures: Normal.
 
EXTENSOR MECHANISM:
Distal quadriceps tendon: No tendinosis or tear.
Patella tendon: No tendinosis or tear.
Medial patellofemoral ligament (MPFL) and patellar retinacula: Normal.
Patellofemoral tracking: No edema in superolateral Hoffa's fat pad. No lateral patellar tilt or translation. No patella alta. Normal patellar and trochlear morphology. Normal tibial tubercle-trochlear groove (TT-TG) distance measuring 17 mm.
 
MUSCLES/TENDONS:
No acute muscle strain or muscle atrophy. Visualized tendons not previously discussed are intact without tendinosis or tear.
 
SOFT TISSUES:
Nonspecific prepatellar and scattered subcutaneous edema about the knee. No soft tissue mass. Normal popliteal neurovascular bundle.
 
IMPRESSION:
1.	Extensive complete tearing of the body, posterior horn, and posterior root medial meniscus with severe extrusion.
2.	Tricompartmental osteoarthritis worse in the medial compartment.""",
        """{
            "ambiguous_laterality": false,
            "findings": {
                "ACL": {"state": "explicit_negative", "evidence": "Normal with both bundles intact"},
                "MCL": {"state": "explicit_negative", "evidence": "Medial collateral ligament (MCL): Normal"},
                "Medial Meniscus": {"state": "explicit_positive", "evidence": "Extensive complete tearing of the body, posterior horn, and posterior root medial meniscus"},
                "Lateral Meniscus": {"state": "explicit_negative", "evidence": "Degenerative signal throughout the lateral meniscus without surfacing tear"},
                "Medial OA": {"state": "explicit_positive", "evidence": "Full thickness cartilage loss along the medial femoral condyle and medial tibial plateau"},
                "Lateral OA": {"state": "explicit_positive", "evidence": "Mild cartilage heterogeneity and fissuring"},
                "PF OA": {"state": "explicit_positive", "evidence": "High-grade cartilage loss along the medial patellar facet"},
                "Effusion": {"state": "explicit_positive", "evidence": "Small joint effusion"},
                "Synovitis": {"state": "unmentioned", "evidence": null},
                "Baker's": {"state": "explicit_negative", "evidence": "No Baker's cyst"},
                "Contusion": {"state": "explicit_negative", "evidence": "No acute fracture or bone bruise"},
                "Fracture": {"state": "explicit_negative", "evidence": "No acute fracture"}
            }
        }""",
    ),
]



# =========================================================
# Extraction
# =========================================================

import json, re
from typing import Literal
from pydantic import BaseModel, Field

 
JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

class _FINDINGS(BaseModel):
    model_config = {"extra": "forbid"}
    state: Literal[
        "explicit_positive", "explicit_negative",
        "hedged_positive", "hedged_negative",
        "unmentioned",
    ]
    evidence: str | None

class VALID_FINDINGS(BaseModel):
    model_config = {"extra": "forbid"}
    ACL: _FINDINGS
    MCL: _FINDINGS
    Medial_Meniscus: _FINDINGS = Field(alias="Medial Meniscus")
    Lateral_Meniscus: _FINDINGS = Field(alias="Lateral Meniscus")
    Medial_OA: _FINDINGS = Field(alias="Medial OA")
    Lateral_OA: _FINDINGS = Field(alias="Lateral OA")
    PF_OA: _FINDINGS = Field(alias="PF OA")
    Effusion: _FINDINGS
    Synovitis: _FINDINGS
    Bakers: _FINDINGS = Field(alias="Baker's")
    Contusion: _FINDINGS
    Fracture: _FINDINGS

class Format(BaseModel):
    model_config = {
        "extra": "forbid",
        "populate_by_name": False,
    }
    ambiguous_laterality: bool
    findings: VALID_FINDINGS

def validate_schema(raw_text: str) -> dict | None:
    """Ensure that the LLM output is in the correct format."""
    try:
        text = raw_text.strip()
        match = JSON_RE.search(text)
        if match:
            text = match.group(0)
        parsed = json.loads(text)
        Format.model_validate(parsed)
        return parsed
    except Exception:
        raise ValueError("schema validation failed")


def load_done_ids(output_path: Path) -> set:
    """
    Return IDs already present in the output file,
    so they can be skipped when resuming processing.
    """
    done = set()
    if output_path.is_file():
        with open(output_path, "r") as f:
            for line in f:
                try:
                    row = json.loads(line)
                    done.add(row["StudyInstanceUID"])
                except (json.JSONDecodeError, KeyError):
                    continue
    return done

# =========================================================
# vLLM engine
# =========================================================

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer


def load_engine():
    """INITIALIZATION"""
    llm = LLM(
        model=MODEL_NAME,
        dtype="float16", 
            # T4 GPU old architecture : native BF16 is not supported
        quantization=None,   
        gpu_memory_utilization=0.9,  
        tensor_parallel_size=1, 
            # One GPU per process (run 2 independent processes when required)
        enable_prefix_caching=True,  
            # Reuse the KV cache for the identical PREFIX across requests;
            # Given our large system prompt and few-shot examples against each report,
            # this can significantly reduce redundant computation
        max_model_len=MAX_MODEL_LEN, max_seq_len_to_capture=MAX_SEQ_LEN,
        disable_log_stats=True, 
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    return llm, tokenizer


PREFIX = [{"role": "system", "content": SYSTEM_PROMPT}] 
for example_report, example_json in FEW_SHOT_EXAMPLES:
    PREFIX.extend([
        {"role": "user", "content": f'Report text:\n"""\n{example_report}\n"""'},
        {"role": "assistant", "content": example_json}
    ])

def build_prompts(tokenizer, report_texts: list[str]) -> list[str]:
    """
    PREFIX remains the same throughout the session,
    only the current report changes per call.
    """
    return [
        tokenizer.apply_chat_template(
            PREFIX + [{"role": "user", "content": f'Report text:\n"""\n{report}\n"""'}], 
            tokenize=False, 
            # returns string;
            # tokenization will be handled by vLLM
            add_generation_prompt=True, 
            enable_thinking=ENABLE_THINKING,
        ) 
        for report in report_texts
    ]


def run_batch(llm, prompts: list[str], max_tokens: int) -> list:
    """vLLM pass"""
    sampling_params = SamplingParams(
        # deterministic output for audit purpose
        temperature=0,      
        max_tokens=max_tokens,
    )
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    return [
        {
            "raw_text": out.outputs[0].text,
            "truncated": out.outputs[0].finish_reason == "length",
            # If any output is truncated due to max_tokens,
            # store the flag as an indicator for downstream retry
        }
        for out in outputs
    ]

# =========================================================
# Main loop with checkpoint/resume
# =========================================================

import argparse, time
from typing import TextIO


def process_batch(
        llm, tokenizer, # see load_engine(...)
        batch_df: pd.DataFrame, out_f: TextIO, 
    ) -> int:
    study_ids = batch_df["StudyInstanceUID"].tolist()
    report_texts = batch_df["Report"].tolist()
    prompts = build_prompts(tokenizer, report_texts)
    n_success = len(study_ids)

    # vLLM pass
    gen_results = run_batch(llm, prompts, MAX_NEW_TOKENS)

    # retry pass: only for whichever requests in this batch actually got truncated 
    truncated_idx = [i for i, r in enumerate(gen_results) if r["truncated"]]
    if truncated_idx:
        retry_prompts = [prompts[i] for i in truncated_idx]
        retry_results = run_batch(llm, retry_prompts, RETRY_MAX_NEW_TOKENS)
        for local_i, orig_i in enumerate(truncated_idx):
            gen_results[orig_i] = retry_results[local_i]

    # validate the JSON before returning
    for study_id, gen in zip(study_ids, gen_results):
        record = {"StudyInstanceUID": study_id, "parse_ok":True}
        try:
            record["out"] = validate_schema(gen["raw_text"])
        except ValueError as e:
            record["out"] = gen["raw_text"]
            n_success -= 1
            record["parse_ok"] = False
            record["error"] = str(e)
            # the failed cases (if any) require manual review;
            # simply running them one more time with the same configuration
            # will not provide any benefit in a zero-temperature setup.

        out_f.write(json.dumps(record) + "\n")

    # flush the batch to disk
    out_f.flush()

    return n_success




def main():
    """THE MAIN LOOP"""

    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--limit", type=int, default=None,
        help="process only N studies (for a quick smoke test)"
    )
    parser.add_argument(
        "--gold_only", action="store_true",
        help="process only gold-labelled studies"
    )
    parser.add_argument(
        "--parity", choices=["all", "even", "odd"], default="all",
        help="process only even/odd-positioned rows — for splitting work across two GPUs running as separate processes"
    )
    args = parser.parse_args()

    # source data
    df = pd.read_csv(TRAIN_CSV)

    # destination path
    output_path = Path(OUTPUT_JSONL)
    if args.parity != "all":
        output_path = output_path.with_stem(f"{output_path.stem}_{args.parity}")
    done_ids = df["StudyInstanceUID"].isin(
        load_done_ids(output_path)
    )
    print(f"Resuming: {done_ids.sum()} studies already extracted, skipping those.")

    # filter pending tasks
    if args.gold_only:
        gold_mask = df[FINDINGS].notna().any(axis=1)
        todo = df[gold_mask & ~done_ids]
    else:
        todo = df[~done_ids]

    # split by odd/even row position, 
    # so two concurrent processes never race for the same study.
    if args.parity != "all":
        remainder = (todo.index % 2).astype(bool) 
        if args.parity == "odd":
            todo = todo[remainder]
        else: todo = todo[~remainder]

    # filter only first few rows (for a quick smoke test)
    if args.limit:
        todo = todo.head(args.limit)
        
    print(f"TODO: {len(todo)} studies to process this session")

    # initialization
    llm, tokenizer = load_engine()

    # main iterations
    start_time = time.time()
    n_processed, n_success = 0, 0
    N = len(todo)

    with open(output_path, "a") as out_f:
        for batch_start in range(0, N, BATCH_SIZE):
            batch_t0 = time.time()
            elapsed = batch_t0 - start_time
            if elapsed > TIME_BUDGET_SECONDS:
                print(
                    f"""
                    Time budget reached ({elapsed/3600:.2f}h) — stopping cleanly.
                    Rerun this script to resume from checkpoint.
                    """
                )
                break

            batch_df = todo.iloc[batch_start:batch_start + BATCH_SIZE] # [start, end)
            n_success += process_batch(
                llm, tokenizer, 
                batch_df, 
                out_f
            )

            current_batch = len(batch_df) # less than or equal to BATCH_SIZE
            n_processed += current_batch
            batch_seconds = time.time() - batch_t0

            print(
                f"""
                [{n_processed}/{N}] batch of {current_batch} in {batch_seconds:.1f}s.
                parse failures so far: {n_processed - n_success} | est. full run at this rate: {(batch_seconds / 3600) * (N / current_batch):.1f}h.
                """
            )

    print(
        f"""
        Done for this session. Success: [{n_success}/{N}]
        Output: {output_path}.
        """
    )


if __name__ == "__main__":
    main()



Overwriting /kaggle/working/report_labeler.py


In [4]:
import os
path = "/kaggle/working/report_labels_raw.jsonl"
if os.path.exists(path):
    os.remove(path)
    pass

In [ ]:
# The provided radiology reports are multilingual, including English, Spanish, Dutch, Greek, Turkish, and other languages.
# Introducing an intermediate translation step adds an additional source of error and may degrade the reliability of the extracted labels.

# Instead, use an instruction-tuned multilingual LLM to directly infer the `TARGETS` from the original reports.
# Calibrate and validate the LLM on the 58 labelled cases before applying it to the larger unlabelled dataset.

# ref:
# - https://aclanthology.org/anthology-files/anthology-files/pdf/clinicalnlp/2026.clinicalnlp-1.35.pdf
# - https://www.sciencedirect.com/science/article/pii/S0720048X25004279

!python /kaggle/working/report_labeler.py --gold_only